# Colab bootstrap — realized-volatility-prediction

Thin launcher, not an experiment notebook (manifesto: shared `src/` library, thin notebooks). Clones the repo, installs the delta packages Colab doesn't already ship, then shells out to the Stage 1 CLI driver (`scripts/run_stage1.py`). All logic lives in `src/`; this notebook only wires up the environment.

Re-run the clone cell any time to pull the latest `main` before a new run.

In [1]:
import os

REPO_URL = "https://github.com/Stridasaurus/realized-volatility-prediction.git"
REPO_DIR = "realized-volatility-prediction"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

Cloning into 'realized-volatility-prediction'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 142 (delta 43), reused 123 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 655.14 KiB | 11.70 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/realized-volatility-prediction


In [2]:
# Unpinned deltas only — Colab already ships numpy/pandas/scipy/torch, and hard-pinning
# the local dev env's versions here risks forcing a slow/breaking reinstall of that stack.
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.8 MB/s eta 0:00:00


In [3]:
!python -m pytest -q

........................................................................ [ 62%]
............................................                             [100%]
116 passed in 21.56s


## Stage 1: tune

Optuna search on TV/h=1, train/val only. Writes the frozen HP set into `configs/default.yaml` (review the diff and commit it — that commit pre-registers the frozen control per manifesto s7). Use `--dry-run` first to preview without touching the config.

In [4]:
!python scripts/run_stage1.py tune --n-trials 50 --dry-run

tuning on rv_tv h=1, 50 trials, train/val only ...
[I 2026-07-05 18:49:21,743] A new study created in memory with name: no-name-571fd9b6-b975-4a84-823f-593a47fdac9d
[I 2026-07-05 18:49:55,950] Trial 0 finished with value: 0.5211214423179626 and parameters: {'hidden': 36, 'layers': 2, 'dropout': 0.30138168803582194, 'lr': 0.001229607110732572, 'weight_decay': 1.3130280280658579e-06, 'batch_size': 128, 'seq_len': 64}. Best is trial 0 with value: 0.5211214423179626.
[I 2026-07-05 18:50:18,552] Trial 1 finished with value: 0.5002856850624084 and parameters: {'hidden': 22, 'layers': 2, 'dropout': 0.26444745987645224, 'lr': 0.001368009527972693, 'weight_decay': 0.0004246031301768212, 'batch_size': 64, 'seq_len': 56}. Best is trial 1 with value: 0.5002856850624084.
[I 2026-07-05 18:50:32,422] Trial 2 finished with value: 0.44722041487693787 and parameters: {'hidden': 68, 'layers': 2, 'dropout': 0.489309171116382, 'lr': 0.00396567508177101, 'weight_decay': 2.029536243043472e-06, 'batch_size': 

## Stage 1: compare

RV-only LSTM vs HAR, walk-forward over the frozen retrain folds, both TV horizons. Reads whatever HP set is currently committed in `configs/default.yaml` — run `tune` (without `--dry-run`) and commit the config change first.

In [5]:
!python scripts/run_stage1.py compare --seeds headline

--- TV h=1 ---
  scoring har ...
  training lstm_rvonly (RV-only: tiers=('t1',)) ...
--- TV h=5 ---
  scoring har ...
  training lstm_rvonly (RV-only: tiers=('t1',)) ...

saved run: /content/realized-volatility-prediction/results/stage1_net_vs_har/20260705T213320Z_dd53da92

=== Stage 1 headline (pure replay of 20260705T213320Z_dd53da92) ===

rv_tv h=1   (QLIKE lower is better)
      model  qlike      rmse  bind%    n  DM stat  p vs HAR  seed agree%
lstm_rvonly 0.4682 0.0005062      0 1630   -1.124     0.261           90
        har 0.4766 0.0005137      0 1630        —         —            —

rv_tv h=5   (QLIKE lower is better)
      model  qlike      rmse  bind%    n  DM stat  p vs HAR  seed agree%
lstm_rvonly 0.2819 0.0004222      0 1626  -0.8252    0.4094           60
        har 0.2877 0.0004022      0 1626        —         —            —

Edge claim requires DM p<0.05 on the ensemble-mean AND same-sign loss differential in >=80% of seeds (manifesto S6).
